<a href="https://colab.research.google.com/github/DrDavidL/learning-dhds/blob/main/DHDS_Session2_CKD_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 HS-DHDS Session 2: Building a CKD Progression Predictor

**Renal Module | Phase 1b**

---

## Learning Objectives

In this hands-on session, you will:

1. **Generate synthetic CKD patient data** with clinically relevant features
2. **Train a classification model** (Random Forest) to predict CKD progression
3. **Construct and interpret a confusion matrix**
4. **Calculate key metrics**: Sensitivity, Specificity, PPV, NPV, Accuracy
5. **Plot ROC curve** and calculate AUROC and AUPRC
6. **Examine SHAP values** to understand feature contributions

---

## ⚠️ Critical Reminders

### Core Principle
> **If you can't verify it independently, don't use it clinically.**

### Data Security
| ✅ This Notebook | ❌ Never in Colab |
|------------------|-------------------|
| Synthetic data only | Real patient data |
| Learning & code development | PHI of any kind |
| Cloud-based exploration | Production analysis |

**Mantra**: *Cloud for learning, local for real data*

---

## Step 0: Install and Import Libraries

First, let's install SHAP (not pre-installed in Colab) and import all necessary libraries.

In [ ]:
# Install SHAP for model explainability
!pip install shap -q

print("✅ SHAP installed successfully!")

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve,
    average_precision_score, RocCurveDisplay,
    PrecisionRecallDisplay
)

# Explainability
import shap

# Settings
np.random.seed(42)  # For reproducibility
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("✅ All libraries imported successfully!")
print(f"   NumPy: {np.__version__}")
print(f"   Pandas: {pd.__version__}")

---

## Step 1: Generate Synthetic CKD Patient Data

We'll create a realistic synthetic dataset with 300 patients. The outcome (`ckd_progression`) indicates whether a patient's CKD stage advances within 2 years.

### Dataset Variables

| Variable | Description | Range |
|----------|-------------|-------|
| patient_id | Unique identifier | CKD001-CKD300 |
| age | Patient age (years) | 40-85 |
| sex | Biological sex | Male/Female |
| baseline_egfr | Baseline eGFR (mL/min/1.73m²) | 15-90 |
| acr | Albumin-to-creatinine ratio (mg/g) | 10-3000 |
| diabetes | Diabetes mellitus | Yes/No |
| hypertension | Hypertension diagnosis | Yes/No |
| systolic_bp | Systolic BP (mmHg) | 100-180 |
| hba1c | Hemoglobin A1c (%) | 5.0-12.0 |
| **ckd_progression** | **OUTCOME** | 0/1 |

In [ ]:
def generate_synthetic_ckd_data(n_patients=300, progression_rate=0.25):
    """
    Generate synthetic CKD patient data with realistic clinical relationships.

    The outcome (CKD progression) is influenced by:
    - Lower baseline eGFR
    - Higher ACR (proteinuria)
    - Diabetes
    - Poor BP control
    - Higher HbA1c
    """

    # Patient IDs
    patient_ids = [f"CKD{str(i).zfill(3)}" for i in range(1, n_patients + 1)]

    # Demographics
    ages = np.random.randint(40, 86, n_patients)
    sexes = np.random.choice(['Male', 'Female'], n_patients, p=[0.55, 0.45])

    # Comorbidities (with realistic correlations)
    diabetes = np.random.choice([0, 1], n_patients, p=[0.4, 0.6])  # 60% diabetic in CKD population
    hypertension = np.random.choice([0, 1], n_patients, p=[0.15, 0.85])  # 85% hypertensive

    # Clinical values
    # eGFR: lower in older patients and diabetics
    baseline_egfr = np.clip(
        90 - (ages - 40) * 0.5 - diabetes * 15 + np.random.normal(0, 12, n_patients),
        15, 90
    )

    # ACR: higher in diabetics
    acr = np.clip(
        np.exp(np.random.normal(4, 1.2, n_patients) + diabetes * 1.5),
        10, 3000
    )

    # Blood pressure: higher if hypertensive, poorly controlled
    systolic_bp = np.clip(
        120 + hypertension * 25 + np.random.normal(0, 15, n_patients),
        100, 180
    )

    # HbA1c: higher if diabetic
    hba1c = np.clip(
        5.5 + diabetes * 2.5 + np.random.normal(0, 0.8, n_patients),
        5.0, 12.0
    )

    # Generate outcome based on risk factors
    # Create a risk score (higher = more likely to progress)
    risk_score = (
        -0.05 * baseline_egfr +      # Lower eGFR = higher risk
        0.001 * acr +                 # Higher ACR = higher risk
        0.8 * diabetes +              # Diabetes increases risk
        0.02 * systolic_bp +          # Higher BP = higher risk
        0.3 * hba1c +                 # Higher HbA1c = higher risk
        0.01 * ages +                 # Older age = slightly higher risk
        np.random.normal(0, 1, n_patients)  # Random noise
    )

    # Convert to probability using sigmoid
    risk_prob = 1 / (1 + np.exp(-0.5 * (risk_score - np.median(risk_score))))

    # Adjust threshold to achieve target progression rate
    threshold = np.percentile(risk_prob, (1 - progression_rate) * 100)
    ckd_progression = (risk_prob > threshold).astype(int)

    # Create DataFrame
    df = pd.DataFrame({
        'patient_id': patient_ids,
        'age': ages,
        'sex': sexes,
        'baseline_egfr': np.round(baseline_egfr, 1),
        'acr': np.round(acr, 0).astype(int),
        'diabetes': ['Yes' if d == 1 else 'No' for d in diabetes],
        'hypertension': ['Yes' if h == 1 else 'No' for h in hypertension],
        'systolic_bp': np.round(systolic_bp, 0).astype(int),
        'hba1c': np.round(hba1c, 1),
        'ckd_progression': ckd_progression
    })

    return df

# Generate the dataset
df = generate_synthetic_ckd_data(n_patients=300, progression_rate=0.25)

print("✅ Synthetic CKD dataset generated!")
print(f"\n📊 Dataset shape: {df.shape[0]} patients × {df.shape[1]} variables")
print(f"\n🎯 Outcome distribution:")
print(df['ckd_progression'].value_counts())
print(f"\n   Progression rate: {df['ckd_progression'].mean():.1%}")

In [ ]:
# View first 10 patients
print("📋 First 10 patients:")
df.head(10)

In [ ]:
# Summary statistics
print("📈 Summary Statistics:")
df.describe()

---

## Step 2: Data Preparation

Before training our model, we need to:
1. Convert categorical variables to numeric (one-hot encoding)
2. Split into training (70%) and test (30%) sets

In [ ]:
# Prepare features for modeling
# Convert categorical variables to numeric

df_model = df.copy()

# One-hot encode categorical variables
df_model['sex_male'] = (df_model['sex'] == 'Male').astype(int)
df_model['diabetes_yes'] = (df_model['diabetes'] == 'Yes').astype(int)
df_model['hypertension_yes'] = (df_model['hypertension'] == 'Yes').astype(int)

# Select features for the model
feature_columns = [
    'age', 'sex_male', 'baseline_egfr', 'acr',
    'diabetes_yes', 'hypertension_yes', 'systolic_bp', 'hba1c'
]

X = df_model[feature_columns]
y = df_model['ckd_progression']

print("✅ Features prepared!")
print(f"\n📊 Feature matrix shape: {X.shape}")
print(f"\n🔢 Features used:")
for i, col in enumerate(feature_columns, 1):
    print(f"   {i}. {col}")

In [ ]:
# Split into training and test sets
# Rationale for 30% split:
# 1. Small dataset (300 patients): We need a large enough test set (N=90) to get
#    reliable performance estimates. If the test set is too small, metrics like
#    Sensitivity can fluctuate wildly with just 1-2 incorrect predictions.
# 2. General Rule:
#    - Large data (>10k): 80/20 or 90/10 split is common.
#    - Small data (<1k): 70/30 or 80/20 is preferred to ensure statistical stability.
#    - Key goal: The test set must have enough "events" (outcomes) to be meaningful.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,  # 30% for testing
    random_state=42,
    stratify=y  # Maintain outcome proportions
)

print("✅ Data split complete!")
print(f"\n📊 Training set: {len(X_train)} patients ({len(X_train)/len(X):.0%})")
print(f"   - Progressors: {y_train.sum()} ({y_train.mean():.1%})")
print(f"   - Non-progressors: {len(y_train) - y_train.sum()} ({1-y_train.mean():.1%})")
print(f"\n📊 Test set: {len(X_test)} patients ({len(X_test)/len(X):.0%})")
print(f"   - Progressors: {y_test.sum()} ({y_test.mean():.1%})")
print(f"   - Non-progressors: {len(y_test) - y_test.sum()} ({1-y_test.mean():.1%})")

---

## Step 3: Train the Random Forest Model

We'll use a Random Forest classifier — an ensemble method that builds multiple decision trees and averages their predictions. It's widely used in clinical prediction models due to its:
- Ability to capture non-linear relationships
- Robustness to outliers
- Built-in feature importance

In [ ]:
# Train Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,      # Number of trees
    max_depth=10,          # Prevent overfitting
    min_samples_split=10,  # Minimum samples to split
    random_state=42,
    n_jobs=-1              # Use all CPU cores
)

# Fit the model
rf_model.fit(X_train, y_train)

print("✅ Random Forest model trained!")
print(f"\n🌲 Model parameters:")
print(f"   - Number of trees: {rf_model.n_estimators}")
print(f"   - Max depth: {rf_model.max_depth}")
print(f"   - Features used: {rf_model.n_features_in_}")

In [ ]:
# Generate predictions
y_pred = rf_model.predict(X_test)           # Binary predictions (0/1)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]  # Probability of progression

print("✅ Predictions generated!")
print(f"\n📊 Prediction summary:")
print(f"   - Predicted progressors: {y_pred.sum()}")
print(f"   - Predicted non-progressors: {len(y_pred) - y_pred.sum()}")

---

## Step 4: Confusion Matrix and Key Metrics

The confusion matrix is the foundation of all classification metrics. Every prediction falls into one of four categories:

|  | Actually Positive | Actually Negative |
|--|-------------------|-------------------|
| **Predicted Positive** | ✅ True Positive (TP) | ❌ False Positive (FP) |
| **Predicted Negative** | ❌ False Negative (FN) | ✅ True Negative (TN) |

In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("📊 CONFUSION MATRIX")
print("=" * 50)
print(f"\n                    Actual Positive    Actual Negative")
print(f"Predicted Positive       TP = {tp}             FP = {fp}")
print(f"Predicted Negative       FN = {fn}             TN = {tn}")
print("\n" + "=" * 50)

In [ ]:
# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))

# Create heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Progression', 'Progression'],
    yticklabels=['No Progression', 'Progression'],
    annot_kws={'size': 20},
    ax=ax
)

# Add labels for TP, TN, FP, FN
ax.text(0.5, 0.7, 'TN', ha='center', va='bottom', fontsize=12, color='gray', transform=ax.transData)
ax.text(1.5, 0.7, 'FP', ha='center', va='bottom', fontsize=12, color='gray', transform=ax.transData)
ax.text(0.5, 1.7, 'FN', ha='center', va='bottom', fontsize=12, color='gray', transform=ax.transData)
ax.text(1.5, 1.7, 'TP', ha='center', va='bottom', fontsize=12, color='white', transform=ax.transData)

ax.set_xlabel('Actual', fontsize=14)
ax.set_ylabel('Predicted', fontsize=14)
ax.set_title('Confusion Matrix: CKD Progression Prediction', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate all key metrics
sensitivity = tp / (tp + fn)  # True Positive Rate / Recall
specificity = tn / (tn + fp)  # True Negative Rate
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0  # Positive Predictive Value / Precision
npv = tn / (tn + fn) if (tn + fn) > 0 else 0  # Negative Predictive Value
accuracy = (tp + tn) / (tp + tn + fp + fn)

print("📈 KEY PERFORMANCE METRICS")
print("=" * 60)
print(f"\n{'Metric':<25} {'Value':>10} {'Interpretation'}")
print("-" * 60)
print(f"{'Sensitivity (Recall)':<25} {sensitivity:>10.1%}   Of progressors, {sensitivity:.0%} were caught")
print(f"{'Specificity':<25} {specificity:>10.1%}   Of non-progressors, {specificity:.0%} correctly ruled out")
print(f"{'PPV (Precision)':<25} {ppv:>10.1%}   Of positive predictions, {ppv:.0%} truly progressed")
print(f"{'NPV':<25} {npv:>10.1%}   Of negative predictions, {npv:.0%} truly didn't progress")
print(f"{'Accuracy':<25} {accuracy:>10.1%}   Overall correct predictions")
print("\n" + "=" * 60)

### 🚨 The Imbalanced Data Problem: Why Accuracy Can Be Misleading

Let's demonstrate why accuracy is a poor metric for rare outcomes by comparing our model to a "trivial" model that predicts **no progression for everyone**.

In [ ]:
# Compare to a trivial "always negative" model
y_pred_trivial = np.zeros_like(y_test)  # Predict no progression for everyone

# Trivial model metrics
cm_trivial = confusion_matrix(y_test, y_pred_trivial)
tn_t, fp_t, fn_t, tp_t = cm_trivial.ravel()

accuracy_trivial = (tp_t + tn_t) / len(y_test)
sensitivity_trivial = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0

print("⚠️  THE IMBALANCED DATA PROBLEM")
print("=" * 70)
print("\nComparing our Random Forest model to a TRIVIAL model that always")
print("predicts 'No Progression' for every patient:")
print("\n" + "-" * 70)
print(f"{'Metric':<25} {'Random Forest':>15} {'Trivial Model':>15} {'Better?':>12}")
print("-" * 70)
print(f"{'Accuracy':<25} {accuracy:>15.1%} {accuracy_trivial:>15.1%} {'⚠️ Trivial!' if accuracy_trivial > accuracy else '✅ RF':>12}")
print(f"{'Sensitivity':<25} {sensitivity:>15.1%} {sensitivity_trivial:>15.1%} {'✅ RF':>12}")
print(f"{'Catches progressors?':<25} {'Yes':>15} {'NO (0%)':>15} {'✅ RF':>12}")
print("-" * 70)
print("\n🎯 KEY INSIGHT:")
print(f"   The trivial model achieves {accuracy_trivial:.0%} accuracy by predicting")
print(f"   'No Progression' for everyone, but catches ZERO actual progressors!")
print(f"   This is why ACCURACY is MISLEADING for rare outcomes.")
print("\n   ➡️  Use AUROC, AUPRC, or Sensitivity/Specificity instead!")

---

## Step 5: ROC Curve, AUROC, and AUPRC

### Understanding the Metrics

| Metric | Measures | Baseline | Best For |
|--------|----------|----------|----------|
| **AUROC** | Discrimination (ranking) | 0.5 (random) | General performance |
| **AUPRC** | Precision-Recall trade-off | = Prevalence | **Rare outcomes** |

### ⚠️ Critical Distinction: AUROC ≠ Calibration

A high AUROC means the model **ranks** patients well (high-risk patients truly have higher event rates). But it does **NOT** mean the predicted probabilities are accurate!

Example: A model might predict "30% risk" when the true risk is only 10%.

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds_roc = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Calculate Precision-Recall curve
precision, recall, thresholds_pr = precision_recall_curve(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

# Baseline for AUPRC = prevalence
prevalence = y_test.mean()

print("📊 DISCRIMINATION METRICS")
print("=" * 50)
print(f"\n   AUROC:  {roc_auc:.3f}")
print(f"   AUPRC:  {pr_auc:.3f}")
print(f"\n   Interpretation:")
if roc_auc >= 0.8:
    print(f"   ✅ AUROC = {roc_auc:.2f} indicates GOOD discrimination")
elif roc_auc >= 0.7:
    print(f"   ⚠️ AUROC = {roc_auc:.2f} indicates ACCEPTABLE discrimination")
else:
    print(f"   ❌ AUROC = {roc_auc:.2f} indicates POOR discrimination")

print(f"\n   📌 AUPRC baseline = prevalence = {prevalence:.2f}")
print(f"      Our AUPRC ({pr_auc:.2f}) vs baseline ({prevalence:.2f}): {pr_auc/prevalence:.1f}x better than random")

In [ ]:
# Plot ROC and PR curves side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax1 = axes[0]
ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'Random Forest (AUC = {roc_auc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier (AUC = 0.5)')
ax1.fill_between(fpr, tpr, alpha=0.2)

ax1.set_xlabel('1 - Specificity (False Positive Rate)', fontsize=12)
ax1.set_ylabel('Sensitivity (True Positive Rate)', fontsize=12)
ax1.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right', fontsize=10)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1.02])

# Add grid lines at 0.2 intervals
ax1.set_xticks(np.arange(0, 1.1, 0.2))
ax1.set_yticks(np.arange(0, 1.1, 0.2))
ax1.grid(True, alpha=0.3)

# Precision-Recall Curve
ax2 = axes[1]
ax2.plot(recall, precision, 'g-', linewidth=2, label=f'Random Forest (AUPRC = {pr_auc:.3f})')
ax2.axhline(y=prevalence, color='gray', linestyle='--', linewidth=1.5,
            label=f'Baseline (Prevalence = {prevalence:.3f})')
ax2.fill_between(recall, precision, alpha=0.2, color='green')

ax2.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax2.set_ylabel('Precision (PPV)', fontsize=12)
ax2.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1.02])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Note: For AUPRC, the baseline is the prevalence (not 0.5 like AUROC).")
print(f"   A random classifier would achieve AUPRC = {prevalence:.3f}")

---

## Step 6: Model Explainability with SHAP

**SHAP (SHapley Additive exPlanations)** values quantify how each feature contributes to individual predictions.

### How to Interpret SHAP Values:

| SHAP Value | Meaning |
|------------|--------|
| **Positive (+)** | Feature **increases** predicted risk |
| **Negative (−)** | Feature **decreases** predicted risk |
| **Magnitude** | Larger |SHAP| = greater influence |

### Why Explainability Matters:
- Builds clinician **trust** in model predictions
- Helps identify potential **errors** or biases
- Enables **communication** of risk factors to patients
- Increasingly **required by FDA** for clinical AI approval

In [ ]:
# Create SHAP explainer
print("⏳ Calculating SHAP values (this may take a moment)...")

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# For binary classification, we want the SHAP values for the positive class (progression)
if isinstance(shap_values, list):
    shap_values_positive = shap_values[1]  # Class 1 (progression)
else:
    shap_values_positive = shap_values

print("✅ SHAP values calculated!")

In [ ]:
# Global Feature Importance (Summary Plot)
import warnings

print("📊 GLOBAL FEATURE IMPORTANCE")
print("="*50)
print("\nThis shows which features are most important OVERALL for predicting Progression:")

# Fix: Ensure we are plotting only the positive class (Progression)
# shap_values_positive might be 3D (samples, features, classes) -> we want 2D (samples, features)
shap_vals_for_plot = shap_values_positive
if len(shap_vals_for_plot.shape) == 3:
    shap_vals_for_plot = shap_vals_for_plot[:, :, 1]  # Select Class 1

plt.figure(figsize=(10, 6))

# Suppress FutureWarning from SHAP/NumPy interaction
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    # plot_type="bar" shows the mean absolute SHAP value (magnitude of importance)
    shap.summary_plot(
        shap_vals_for_plot,
        X_test,
        plot_type="bar",
        max_display=len(feature_columns), # Ensure all features are shown
        show=False
    )

plt.title("Feature Importance (mean |SHAP|) - CKD Progression", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Beeswarm Plot - shows direction and magnitude
import warnings

print("📊 SHAP BEESWARM PLOT")
print("="*50)
print("\nEach dot is a patient. Color shows feature value (red=high, blue=low).")
print("Position shows impact on prediction (right=increases risk, left=decreases).")

# Fix: Ensure we are plotting only the positive class (Progression)
shap_vals_for_plot = shap_values_positive
if len(shap_vals_for_plot.shape) == 3:
    shap_vals_for_plot = shap_vals_for_plot[:, :, 1]  # Select Class 1

plt.figure(figsize=(10, 6))

# Suppress FutureWarning from SHAP/NumPy interaction
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    shap.summary_plot(shap_vals_for_plot, X_test, show=False)

plt.title("SHAP Value Impact on Prediction", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Explaining Individual Patient Predictions

Let's examine SHAP values for specific patients to understand **why** the model made its predictions.

In [ ]:
# Find a high-risk patient (predicted progression) and a low-risk patient
high_risk_idx = np.argmax(y_pred_proba)
low_risk_idx = np.argmin(y_pred_proba)

print("🔍 INDIVIDUAL PATIENT EXPLANATIONS")
print("="*70)

# High-risk patient
print(f"\n🔴 HIGH-RISK PATIENT (Predicted risk: {y_pred_proba[high_risk_idx]:.1%})")
print("-"*70)
print("Clinical Features:")
for col in feature_columns:
    print(f"   {col}: {X_test.iloc[high_risk_idx][col]}")
print(f"\nActual outcome: {'PROGRESSED' if y_test.iloc[high_risk_idx] == 1 else 'Did not progress'}")

In [ ]:
# SHAP waterfall plot for high-risk patient
print("\n📊 SHAP Waterfall Plot - High Risk Patient")
print("Shows how each feature pushed the prediction up (red) or down (blue)")

# Prepare base value (handle both scalar and array cases)
base_value = explainer.expected_value
if hasattr(base_value, "__len__") and len(base_value) > 1:
    base_value = base_value[1] # Positive class base value

# Create explanation object for waterfall plot
# Ensure values are 1D for the specific class (index 1 for positive class)
shap_explanation = shap.Explanation(
    values=shap_values_positive[high_risk_idx, :, 1],
    base_values=base_value,
    data=X_test.iloc[high_risk_idx].values,
    feature_names=feature_columns
)

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_explanation, show=False)
plt.title(f"High-Risk Patient (Predicted: {y_pred_proba[high_risk_idx]:.1%})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Low-risk patient
print(f"\n🟢 LOW-RISK PATIENT (Predicted risk: {y_pred_proba[low_risk_idx]:.1%})")
print("-"*70)
print("Clinical Features:")
for col in feature_columns:
    print(f"   {col}: {X_test.iloc[low_risk_idx][col]}")
print(f"\nActual outcome: {'PROGRESSED' if y_test.iloc[low_risk_idx] == 1 else 'Did not progress'}")

# SHAP waterfall plot for low-risk patient
print("\n📊 SHAP Waterfall Plot - Low Risk Patient")

# Prepare base value (handle both scalar and array cases)
base_value = explainer.expected_value
if hasattr(base_value, "__len__") and len(base_value) > 1:
    base_value = base_value[1] # Positive class base value

# Fix: Select SHAP values for the positive class (index 1)
shap_explanation_low = shap.Explanation(
    values=shap_values_positive[low_risk_idx, :, 1],
    base_values=base_value,
    data=X_test.iloc[low_risk_idx].values,
    feature_names=feature_columns
)

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_explanation_low, show=False)
plt.title(f"Low-Risk Patient (Predicted: {y_pred_proba[low_risk_idx]:.1%})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 📋 Summary: Key Takeaways

### What We Built
- A **Random Forest classifier** to predict CKD progression
- Trained on **synthetic data** (cloud-safe!)
- Evaluated using multiple metrics appropriate for clinical prediction

### Critical Lessons

| Concept | Key Point |
|---------|----------|
| **Confusion Matrix** | Foundation for all classification metrics (TP, TN, FP, FN) |
| **Accuracy** | ⚠️ **Misleading for rare outcomes** — trivial models can achieve high accuracy! |
| **AUROC** | Measures **discrimination** (ranking ability), not calibration |
| **AUPRC** | Better for **rare outcomes** — baseline = prevalence, NOT 0.5 |
| **SHAP** | Explains **individual predictions** — essential for clinical trust |

### Core Principle
> **If you can't verify it independently, don't use it clinically.**

### Data Security Reminder
- ✅ **This notebook**: Synthetic data, safe for cloud
- ❌ **Never upload**: Real patient data to Colab or any cloud service
- 📋 **Workflow**: Develop code here → Apply to real data on approved local systems only

In [ ]:
# Final summary
print("\n" + "="*70)
print("📊 FINAL MODEL PERFORMANCE SUMMARY")
print("="*70)
print(f"\n🎯 Dataset: {len(df)} synthetic CKD patients")
print(f"   Outcome prevalence: {y.mean():.1%}")
print(f"\n📈 Model Performance (Test Set, n={len(y_test)}):")
print(f"   • AUROC:       {roc_auc:.3f}  (>0.8 = Good)")
print(f"   • AUPRC:       {pr_auc:.3f}  (baseline = {prevalence:.3f})")
print(f"   • Sensitivity: {sensitivity:.1%}  (of progressors caught)")
print(f"   • Specificity: {specificity:.1%}  (of non-progressors correctly ruled out)")
print(f"   • PPV:         {ppv:.1%}  (of positive predictions correct)")
print(f"   • NPV:         {npv:.1%}  (of negative predictions correct)")

print(f"\n🔍 Top 3 Most Important Features (by mean |SHAP|):")

# Fix: Ensure we are calculating mean over the positive class SHAP values only
if len(shap_values_positive.shape) == 3:
    shap_vals_target = shap_values_positive[:, :, 1]
else:
    shap_vals_target = shap_values_positive

mean_shap = np.abs(shap_vals_target).mean(axis=0)
feature_importance = sorted(zip(feature_columns, mean_shap), key=lambda x: x[1], reverse=True)
for i, (feat, imp) in enumerate(feature_importance[:3], 1):
    print(f"   {i}. {feat}")
print("\n" + "="*70)
print("✅ Session 2 hands-on complete!")
print("="*70)

---

## 🎓 Next Steps & Resources

### To Explore Further:
1. **Modify the model**: Try different algorithms (Logistic Regression, XGBoost)
2. **Tune hyperparameters**: Adjust n_estimators, max_depth
3. **Add features**: Include more clinical variables
4. **Examine calibration**: Plot predicted vs. observed probabilities

### Resources:
- **Course materials**: github.com/DrDavidL/learning-dhds
- **TRIPOD Statement**: tripod-statement.org
- **SHAP Documentation**: shap.readthedocs.io

### Contact:
**David Liebovitz, MD** — DavidL@northwestern.edu